In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, desc
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("Distributed_AI_Crime_Analysis") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session Initialized...")

Spark Session Initialized...


In [23]:
# Make sure to run this cell from the same directory where notebook is stored!
data_path = "../crime---analysis/data/cleaned_crime_data.csv"
df = spark.read.csv(data_path, header=True, inferSchema=True)

# Cast columns to integers
df = df.withColumn("YEAR", col("YEAR").cast("integer")) \
       .withColumn("crime_count", col("crime_count").cast("integer")) \
       .dropna(subset=["crime_count"])

df.createOrReplaceTempView("crime_data")
df.printSchema()
print(f"Total records in distributed dataset: {df.count()}")

root
 |-- STATE/UT: string (nullable = true)
 |-- DISTRICT: string (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- crime_type: string (nullable = true)
 |-- crime_count: integer (nullable = true)

Total records in distributed dataset: 252476


## Distributed Analytical Queries (Spark SQL)

In [24]:
# Query 1: Year-wise Total Crimes in India
year_trend = spark.sql("""
    SELECT YEAR, SUM(crime_count) as total_crimes 
    FROM crime_data 
    GROUP BY YEAR 
    ORDER BY YEAR ASC
""")
year_trend.show(5)

+----+------------+
|YEAR|total_crimes|
+----+------------+
|2001|     2718622|
|2002|     2671436|
|2003|     2593568|
|2004|     2811178|
|2005|     2789854|
+----+------------+
only showing top 5 rows



In [25]:
# Query 2: Top 10 Most Frequent Crime Types
top_crimes = spark.sql("""
    SELECT crime_type, SUM(crime_count) as total_crimes 
    FROM crime_data 
    GROUP BY crime_type 
    ORDER BY total_crimes DESC
    LIMIT 10
""")
top_crimes.show(truncate=False)

+---------------------------------------------------+------------+
|crime_type                                         |total_crimes|
+---------------------------------------------------+------------+
|THEFT                                              |7001060     |
|HURT/GREVIOUS HURT                                 |6743752     |
|OTHER THEFT                                        |4460320     |
|AUTO THEFT                                         |2540740     |
|BURGLARY                                           |2234678     |
|CAUSING DEATH BY NEGLIGENCE                        |2003728     |
|CRUELTY BY HUSBAND OR HIS RELATIVES                |1750402     |
|RIOTS                                              |1549854     |
|CHEATING                                           |1535194     |
|ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY|906310      |
+---------------------------------------------------+------------+



## AI & Machine Learning Component (PySpark MLlib)
To fulfill the AI objective, we will run a distributed `KMeans` clustering algorithm. 
We aggregate states and create a feature vector from their top 5 highest occurring crime types, scaling them before passing them to the clusterer.

In [26]:
top_5_crime_types = [row['crime_type'] for row in top_crimes.take(5)]

# Pivot the dataset logically to create distinct features
features_df = df.filter(col("crime_type").isin(top_5_crime_types)) \
                .groupBy("STATE/UT") \
                .pivot("crime_type") \
                .sum("crime_count") \
                .na.fill(0)

# Vectorize and Scale
assembler = VectorAssembler(inputCols=top_5_crime_types, outputCol="features")
assembled_df = assembler.transform(features_df)

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=True)
scalerModel = scaler.fit(assembled_df)
scaled_df = scalerModel.transform(assembled_df)

# Train K-Means with K=3 (e.g. Risk Levels: High, Medium, Low)
k = 3
kmeans = KMeans(featuresCol="scaledFeatures", predictionCol="cluster", k=k, seed=1)
model = kmeans.fit(scaled_df)
predictions = model.transform(scaled_df)

# Map clusters to Risk Levels based on their centroid sum (Magnitude of crimes)
centers = model.clusterCenters()
magnitudes = {i: float(sum(c)) for i, c in enumerate(centers)}
sorted_clusters = sorted(magnitudes.items(), key=lambda x: x[1], reverse=True)

risk_mapping = {
    sorted_clusters[0][0]: "High Risk",
    sorted_clusters[1][0]: "Medium Risk",
    sorted_clusters[2][0]: "Low Risk"
}

print("\n>>> Risk Level Mapping:")
for c_id, r_label in risk_mapping.items():
    print(f"Cluster {c_id} corresponds to {r_label} (Severity Score: {magnitudes[c_id]:.2f})")

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
risk_udf = udf(lambda c: risk_mapping[c], StringType())

final_predictions = predictions.withColumn("Risk_Level", risk_udf(col("cluster")))




>>> Risk Level Mapping:
Cluster 1 corresponds to High Risk (Severity Score: 11.09)
Cluster 2 corresponds to Medium Risk (Severity Score: 3.38)
Cluster 0 corresponds to Low Risk (Severity Score: -2.77)


In [27]:

print(f"\n>>> All States and their assigned Risk Levels (K={k}):")
final_predictions.select("STATE/UT", "cluster", "Risk_Level").orderBy("cluster", "STATE/UT").show(40, truncate=False)


>>> All States and their assigned Risk Levels (K=3):
+-----------------+-------+-----------+
|STATE/UT         |cluster|Risk_Level |
+-----------------+-------+-----------+
|A & N ISLANDS    |0      |Low Risk   |
|ARUNACHAL PRADESH|0      |Low Risk   |
|ASSAM            |0      |Low Risk   |
|CHANDIGARH       |0      |Low Risk   |
|CHHATTISGARH     |0      |Low Risk   |
|D & N HAVELI     |0      |Low Risk   |
|DAMAN & DIU      |0      |Low Risk   |
|GOA              |0      |Low Risk   |
|HIMACHAL PRADESH |0      |Low Risk   |
|JAMMU & KASHMIR  |0      |Low Risk   |
|JHARKHAND        |0      |Low Risk   |
|KERALA           |0      |Low Risk   |
|LAKSHADWEEP      |0      |Low Risk   |
|MANIPUR          |0      |Low Risk   |
|MEGHALAYA        |0      |Low Risk   |
|MIZORAM          |0      |Low Risk   |
|NAGALAND         |0      |Low Risk   |
|ODISHA           |0      |Low Risk   |
|PUDUCHERRY       |0      |Low Risk   |
|PUNJAB           |0      |Low Risk   |
|SIKKIM           |0      

In [28]:
print("\n=== HIGH RISK STATES ===\n")
final_predictions.filter(col("Risk_Level") == "High Risk").select("STATE/UT").show(40, truncate=False)


=== HIGH RISK STATES ===

+--------------+
|STATE/UT      |
+--------------+
|MADHYA PRADESH|
|MAHARASHTRA   |
|ANDHRA PRADESH|
+--------------+



In [29]:
print("=== MEDIUM RISK STATES ===\n")
final_predictions.filter(col("Risk_Level") == "Medium Risk").select("STATE/UT").show(40, truncate=False)

=== MEDIUM RISK STATES ===

+-------------+
|STATE/UT     |
+-------------+
|WEST BENGAL  |
|RAJASTHAN    |
|GUJARAT      |
|BIHAR        |
|KARNATAKA    |
|UTTAR PRADESH|
|DELHI UT     |
|TAMIL NADU   |
|HARYANA      |
+-------------+



In [30]:
print("=== LOW RISK STATES ===\n")
final_predictions.filter(col("Risk_Level") == "Low Risk").select("STATE/UT").show(40, truncate=False)

=== LOW RISK STATES ===

+-----------------+
|STATE/UT         |
+-----------------+
|SIKKIM           |
|MEGHALAYA        |
|DAMAN & DIU      |
|A & N ISLANDS    |
|GOA              |
|CHHATTISGARH     |
|TRIPURA          |
|HIMACHAL PRADESH |
|CHANDIGARH       |
|JAMMU & KASHMIR  |
|MANIPUR          |
|LAKSHADWEEP      |
|PUDUCHERRY       |
|PUNJAB           |
|NAGALAND         |
|ARUNACHAL PRADESH|
|UTTARAKHAND      |
|MIZORAM          |
|ODISHA           |
|KERALA           |
|ASSAM            |
|D & N HAVELI     |
|JHARKHAND        |
+-----------------+



In [31]:
spark.stop()